# Section 3: Creating a Streaming S3 Ingestion Flow

*Notes:* This notebook sets up a streaming ingestion pipeline from S3 into a Databricks Delta table.
Prerequisites: an accessible S3 bucket, appropriate IAM credentials, and Unity Catalog or workspace privileges.

The detailed background of this code is in this blog:

https://medium.com/@junshan0/creating-a-streaming-s3-ingestion-flow-05dd6e21c7db?sk=a939b393c5b8520e00212938351749c0

*Note:* The blog provides guidance for configuring IAM policies and Databricks storage credentials.

In [ ]:
%sql
CREATE CATALOG IF NOT EXISTS video_ai;
CREATE SCHEMA  IF NOT EXISTS video_ai.bronze;
-- Comment: Creates a Unity Catalog catalog and a bronze schema for raw ingests. Run in a SQL cell.

In [ ]:
%sql
CREATE VOLUME IF NOT EXISTS video_ai.bronze.ingest_state;
-- Comment: A volume to store checkpoint and schema state for the streaming ingestion.

In [ ]:
from pyspark.sql.functions import col, current_timestamp, lit, input_file_name

# Paths and source configuration - replace <bucket> with your S3 bucket name
source_path     = "s3://<bucket>/"
checkpoint_path = "/Volumes/video_ai/bronze/ingest_state/videos_checkpoint"
schema_path     = "/Volumes/video_ai/bronze/ingest_state/videos_schema"

# Configure Auto Loader (cloudFiles) to ingest binary files (video files)
raw = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "binaryFile")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("pathGlobFilter", "*.mp4")          # only pick up video files
    .option("cloudFiles.includeExistingFiles", "true")  # backfill files already in the bucket
    .load(source_path)
)

# binaryFile yields: path, modificationTime, length, content
# Drop `content` (the bytes) and shape the Bronze metadata record.
bronze_df = (
    raw
    .withColumn("file_path", col("path"))
    .withColumn("file_name", col("path").substr(-100, 100))  # or use a regex/expr for the basename
    .withColumn("file_size_bytes", col("length"))
    .withColumn("file_modified_time", col("modificationTime"))
    .withColumn("file_type", lit("mp4"))
    .withColumn("processing_status", lit("INGESTED"))
    .withColumn("source", lit("s3://video-ai-workshop"))
    .withColumn("ingestion_timestamp", current_timestamp())
    .drop("content", "path", "length", "modificationTime")
)

# Comment: Adjust `.option` values for your environment. Use `.option('cloudFiles.backfillInterval', ...)`
# if you need different backfill behavior. Ensure the checkpoint and schema locations are reachable.

In [ ]:
%sql
SELECT
  path                                   AS file_path,
  regexp_extract(path, '([^/]+)$', 1)    AS file_name,
  length                                 AS file_size_bytes,
  modificationTime                       AS file_modified_time,
  'mp4'                                  AS file_type,
  'INGESTED'                             AS processing_status,
  's3://<bucket>'                        AS source,
  current_timestamp()                    AS ingestion_timestamp
FROM read_files(
  's3://<bucket>/',
  format          => 'binaryFile',
  pathGlobFilter  => '*.mp4'
);

-- Comment: This SQL shows the schema that Auto Loader produces for binaryFile reads. Replace <bucket> when running.

In [ ]:
(
    bronze_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)  # availableNow processes all available data and stops
    .toTable("video_ai.bronze.video_files")
)

# Comment: Use `availableNow=True` to run the pipeline once for backfill. For continuous streaming,
# remove `.trigger(availableNow=True)` and start a long-running stream.

In [ ]:
%sql
CREATE OR REFRESH STREAMING TABLE video_ai.bronze.video_files AS
SELECT
  path                                   AS file_path,
  regexp_extract(path, '([^/]+)$', 1)    AS file_name,
  length                                 AS file_size_bytes,
  modificationTime                       AS file_modified_time,
  'mp4'                                  AS file_type,
  'INGESTED'                             AS processing_status,
  's3://video-ai-workshop'               AS source,
  current_timestamp()                    AS ingestion_timestamp
FROM STREAM read_files(
  's3://video-ai-workshop/',
  format          => 'binaryFile',
  pathGlobFilter  => '*.mp4'
);

-- Comment: Creates/refreshes a streaming table backed by the Auto Loader `read_files` source.

In [ ]:
%sql
-- Refresh once (drains any newly arrived files)
REFRESH STREAMING TABLE video_ai.bronze.video_files;

-- Or attach a recurring schedule to the table itself
ALTER STREAMING TABLE video_ai.bronze.video_files
  ADD SCHEDULE CRON '0 */15 * * * ?';   -- every 15 minutes

-- Comment: Scheduling requires Databricks SQL scheduled jobs or managed table scheduling support.